In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

Chargement des données

*************************  
**MOVIES** : liste des films et leurs genres  
*************************

In [4]:
movies = pd.read_csv('movies.csv')

# Remplacement du genre (no genres listed) par NaN
movies['genres'] = movies['genres'].replace('(no genres listed)', np.nan)

# Séparation des genres en listes
movies = movies.join(
    movies['genres'].str.get_dummies(sep='|').add_prefix('genre_')
)

movies = movies.drop(columns=['genres'])

print(movies.shape)

movies.head(2)

(27278, 21)


,movieId,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


*************************  
**RATINGS** : notes des films par les utilisateurs  (0,5 <= note <= 1)
*************************  

In [6]:
ratings = pd.read_csv('ratings.csv')

# le champ timestamp ne nous est pas utile
ratings = ratings.drop(columns=['timestamp'])

display(ratings.head(2))
ratings.info(show_counts=True)

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5


<class 'pandas.DataFrame'>
RangeIndex: 20000263 entries, 0 to 20000262
Data columns (total 3 columns):
 #   Column   Non-Null Count     Dtype  
---  ------   --------------     -----  
 0   userId   20000263 non-null  int64  
 1   movieId  20000263 non-null  int64  
 2   rating   20000263 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 457.8 MB


La moyenne bayésienne par film n'est pas calculée ici parce qu'elle n'est pas utilisée par nos modèles

************************* 
**TAGS** : tag des films par les utilisateurs  
************************* 

In [52]:
tags = pd.read_csv('tags.csv')

tags = tags.dropna()
#tags['timestamp'] = pd.to_datetime(tags['timestamp'], unit='s')
# le champ timestamp ne nous est pas utile
tags = tags.drop(columns=['timestamp'])

display(tags.head())
tags.info()

,userId,movieId,tag
0,18,4141,Mark Waters
1,65,208,dark hero
2,65,353,dark hero
3,65,521,noir thriller
4,65,592,dark hero


<class 'pandas.DataFrame'>
Index: 465548 entries, 0 to 465563
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   userId   465548 non-null  int64
 1   movieId  465548 non-null  int64
 2   tag      465548 non-null  str  
dtypes: int64(2), str(1)
memory usage: 14.2 MB


In [53]:
# dans le dataframe tags nous avons plusieurs lignes pour un film, soit une ligne par tag. On va regrouper les tags en une seule ligne par film
tags = (
    tags
    .groupby('movieId')['tag']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
)
display(tags.head(2))
tags.shape

,movieId,tag
0,1,Watched computer animation Disney animated fea...
1,2,time travel adapted from:book board game child...


(19545, 2)

In [54]:
# Ajout du titre du film au dataframe tags via movieId
tags = tags.merge(movies[['movieId', 'title']], on='movieId', how='left')

# Vérification
tags.head()

,movieId,tag,title
0,1,Watched computer animation Disney animated fea...,Toy Story (1995)
1,2,time travel adapted from:book board game child...,Jumanji (1995)
2,3,old people that is actually funny sequel fever...,Grumpier Old Men (1995)
3,4,chick flick revenge characters chick flick cha...,Waiting to Exhale (1995)
4,5,Diane Keaton family sequel Steve Martin weddin...,Father of the Bride Part II (1995)


************************* 
**GENOME SCORES** : score de pertinence de chaque tag pour chaque film    
************************* 

In [32]:
genome_scores = pd.read_csv('genome-scores.csv')
display(genome_scores.head(2))
genome_scores.info(show_counts=True)

,movieId,tagId,relevance
0,1,1,0.025
1,1,2,0.025


<class 'pandas.DataFrame'>
RangeIndex: 11709768 entries, 0 to 11709767
Data columns (total 3 columns):
 #   Column     Non-Null Count     Dtype  
---  ------     --------------     -----  
 0   movieId    11709768 non-null  int64  
 1   tagId      11709768 non-null  int64  
 2   relevance  11709768 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 268.0 MB


In [33]:
# On définit un seuil de pertinence pour les tags. on ne garde que les tags vraiment représentatifs du film
# c'est à dire avec un score supérieur ou égal au seuil de pertinence
SEUIL_PERTINENCE = 0.5
genome_scores = genome_scores[genome_scores['relevance'] >= SEUIL_PERTINENCE]
genome_scores.shape

(460090, 3)

In [34]:
# on récupère les libellé des tags
genome_tags = pd.read_csv('genome-tags.csv')
display(genome_tags.head(2))
genome_tags.info()

,tagId,tag
0,1,007
1,2,007 (series)


<class 'pandas.DataFrame'>
RangeIndex: 1128 entries, 0 to 1127
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   tagId   1128 non-null   int64
 1   tag     1128 non-null   str  
dtypes: int64(1), str(1)
memory usage: 17.8 KB


In [35]:
# Fusion genome_scores + genome_tags pour obtenir les noms des tags
df = genome_scores.merge(genome_tags, on='tagId', how='left')
display(df.head(2))

,movieId,tagId,relevance,tag
0,1,11,0.5770,3d
1,1,19,0.6705,action


In [36]:
# Pour chaque film : concaténation des tags les plus pertinents. Pour chaque film les tags sont uniques
df = (
    df
    .groupby('movieId')['tag']
    .apply(lambda x: ' '.join(x.astype(str)))
    .reset_index()
)
df.head(2)

,movieId,tag
0,1,3d action adventure affectionate animal movie ...
1,2,action adapted from:book adventure animal movi...


In [57]:
# Ajout du titre du film au dataframe df via movieId
df = df.merge(movies[['movieId', 'title']], on='movieId', how='left')

# Vérification
df.head()

,movieId,tag,title
0,1,3d action adventure affectionate animal movie ...,Toy Story (1995)
1,2,action adapted from:book adventure animal movi...,Jumanji (1995)
2,3,chase comedy crappy sequel destiny family fish...,Grumpier Old Men (1995)
3,4,adultery betrayal chick flick divorce feel goo...,Waiting to Exhale (1995)
4,5,catastrophe chase comedy crappy sequel culture...,Father of the Bride Part II (1995)


************************* 
**EXPORT DES DONNEES**    
************************* 

In [58]:
# dimensions finales des DataFrames
print(f"Movies : {movies.shape}")
print(f"Ratings: {ratings.shape}")
print(f"Tags: {tags.shape}")   
print(f"genome_scores: {genome_scores.shape}")
print(f"genome Tags Movies: {df.shape}")      

Movies : (27278, 21)
Ratings: (20000263, 3)
Tags: (19545, 3)
genome_scores: (460090, 3)
genome Tags Movies: (10381, 3)


In [59]:
df.to_csv('genome_tags_movies.csv', index=False)

In [48]:
# liste des films avec les genres vectorizés en colonnes. Identifiant : movieId
movies.to_csv('movies_cleaned.csv', index=False)

# liste des notes des utilisateurs. Identifiant : movieId, userId
ratings.to_csv('ratings_cleaned.csv', index=False)

# tag des utilisateurs par film. Identifiant : movieId
tags.to_csv('tags_cleaned.csv', index=False)

# genome score des tags des films par les utilisateurs. Identifiant : movieId, tagId
genome_scores.to_csv('genome_scores_cleaned.csv', index=False)

# genome tag par film. Identifiant : movieId
df.to_csv('genome_tags_movies.csv', index=False)